# sEMG Prosthetic Gesture Classification
## Notebook 13: Systematic Ablation Studies

**Project:** Machine Learning-Based sEMG Prosthetic Gesture Classification

---

### Research Objective & Scope
Quantify the contribution of individual features, sEMG channels, and feature-family domains
to classifier performance via real ablation experiments: for each configuration, a fresh
CatBoost model (cloned from the fully re-tuned 308-trial hyperparameters) is refit on the
**real** 484,700-row subject-disjoint training split restricted to that configuration's
feature subset, and evaluated on the real 103,709-row held-out test split.

**Provenance note:** this notebook was rewritten to run real full-scale ablation training.
The prior version read pre-computed tables from `scratch/generate_notebook13_ablation.py`,
which trained on tiny, non-representative subsamples (5,000-10,000 rows out of 484,700) --
this produced internally inconsistent baseline accuracy figures (24%, 38%, and 41% for the
same nominal baseline configuration across different tables). Real training was executed via
`13_run_ablation_studies_Colab.ipynb` on Google Colab (T4 GPU, after upgrading CatBoost to
resolve an initial CUDA/driver incompatibility), completing all 20 configurations in
**~16 minutes** (19 configs at ~53s each on GPU; the baseline config alone ran before the GPU
fix, on CPU, taking 90 minutes) -- checkpoints are in
`outputs/ablation_checkpoints_v2/CATBOOST/`.

In [1]:
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

tables_dir = PROJECT_ROOT / 'outputs' / 'tables'
figures_dir = PROJECT_ROOT / 'outputs' / 'figures'
reports_dir = PROJECT_ROOT / 'outputs' / 'reports'
checkpoints_dir = PROJECT_ROOT / 'outputs' / 'ablation_checkpoints_v2' / 'CATBOOST'

for d in (tables_dir, figures_dir, reports_dir):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Checkpoint dir: {checkpoints_dir}")


Project Root: E:\Bio-Mechanics\semg-prosthetic-gesture-classification
Checkpoint dir: E:\Bio-Mechanics\semg-prosthetic-gesture-classification\outputs\ablation_checkpoints_v2\CATBOOST


## Section 1: Checkpoint Integrity Verification
Verifies all 20 real ablation configuration checkpoints (produced by
`13_run_ablation_studies_Colab.ipynb`) are present.

In [2]:
ablation_files = sorted(checkpoints_dir.glob("ablation_*.json"))
print(f"Verified Checkpoints: {len(ablation_files)}/20 ablation configuration files.")
assert len(ablation_files) == 20, "Missing ablation checkpoints -- run 13_run_ablation_studies_Colab.ipynb first."

rows = [json.load(open(f)) for f in ablation_files]
df_all = pd.DataFrame(rows)
baseline = df_all[df_all["Config"] == "baseline_all_50_features"].iloc[0]
baseline_acc = baseline["Accuracy"]
baseline_f1 = baseline["Macro F1"]
print(f"Baseline (all 50 features): Accuracy={baseline_acc:.4f}, Macro F1={baseline_f1:.4f}")


Verified Checkpoints: 20/20 ablation configuration files.
Baseline (all 50 features): Accuracy=0.4551, Macro F1=0.1716


## Section 2: Feature Removal Analysis
Real effect of removing the top-N vs. bottom-N SHAP-ranked features (N = 5, 10, 15, 20) --
each row is a genuinely retrained CatBoost model on the real training split restricted to the
remaining features.

In [3]:
removal_order = [
    ("baseline_all_50_features", "Baseline (All 50 Features)"),
    ("remove_top_5_shap_features", "Remove Top 5 SHAP Features"),
    ("remove_top_10_shap_features", "Remove Top 10 SHAP Features"),
    ("remove_top_15_shap_features", "Remove Top 15 SHAP Features"),
    ("remove_top_20_shap_features", "Remove Top 20 SHAP Features"),
    ("remove_bottom_5_shap_features", "Remove Bottom 5 SHAP Features"),
    ("remove_bottom_10_shap_features", "Remove Bottom 10 SHAP Features"),
    ("remove_bottom_15_shap_features", "Remove Bottom 15 SHAP Features"),
    ("remove_bottom_20_shap_features", "Remove Bottom 20 SHAP Features"),
]
rows_feat = []
for cfg, label in removal_order:
    r = df_all[df_all["Config"] == cfg].iloc[0]
    removed = baseline["Feature Count"] - r["Feature Count"]
    f1_drop = baseline["Macro F1"] - r["Macro F1"]
    rows_feat.append({
        "Configuration": label,
        "Features Removed": int(removed),
        "Features Retained": int(r["Feature Count"]),
        "Accuracy": r["Accuracy"],
        "Macro F1": r["Macro F1"],
        "F1 Drop vs Baseline": f1_drop,
        "Relative F1 Drop (%)": 100 * f1_drop / baseline["Macro F1"] if baseline["Macro F1"] else np.nan,
        "Drop Per Removed Feature": f1_drop / removed if removed else 0.0,
    })
df_feat_removal = pd.DataFrame(rows_feat)
df_feat_removal.to_csv(tables_dir / "feature_removal_summary_v2.csv", index=False)
df_feat_removal.to_markdown(tables_dir / "feature_removal_summary_v2.md", index=False)
display(df_feat_removal)


,Configuration,Features Removed,Features Retained,Accuracy,Macro F1,F1 Drop vs Baseline,Relative F1 Drop (%),Drop Per Removed Feature
0,Baseline (All 50 Features),0,50,0.455120,0.171567,0.000000,0.000000,0.000000
1,Remove Top 5 SHAP Features,5,45,0.448409,0.160611,0.010957,6.386165,0.002191
2,Remove Top 10 SHAP Features,10,40,0.443018,0.152413,0.019154,11.164173,0.001915
3,Remove Top 15 SHAP Features,15,35,0.436018,0.133218,0.038349,22.352321,0.002557
4,Remove Top 20 SHAP Features,20,30,0.429027,0.118360,0.053208,31.012633,0.002660
5,Remove Bottom 5 SHAP Features,5,45,0.454098,0.174293,-0.002725,-1.588388,-0.000545
6,Remove Bottom 10 SHAP Features,10,40,0.455052,0.175108,-0.003540,-2.063570,-0.000354
7,Remove Bottom 15 SHAP Features,15,35,0.452728,0.172342,-0.000775,-0.451775,-0.000052
8,Remove Bottom 20 SHAP Features,20,30,0.453056,0.172882,-0.001314,-0.766135,-0.000066


## Section 3: Channel Efficiency Analysis
Real accuracy/latency trade-off when restricting to the top-K most SHAP-important channels
(K = 8, 6, 4, 2, 1), each a genuinely retrained model.

In [4]:
channel_order = [
    ("baseline_all_50_features", "All Channels (12, 50 features)"),
    ("top_8_channels", "Top 8 Channels"),
    ("top_6_channels", "Top 6 Channels"),
    ("top_4_channels", "Top 4 Channels"),
    ("top_2_channels", "Top 2 Channels"),
    ("single_best_channel", "Single Best Channel"),
]
rows_ch = []
for cfg, label in channel_order:
    r = df_all[df_all["Config"] == cfg].iloc[0]
    rows_ch.append({
        "Configuration": label,
        "Features Used": int(r["Feature Count"]),
        "Accuracy": r["Accuracy"],
        "Accuracy Retention (%)": 100 * r["Accuracy"] / baseline["Accuracy"],
        "Macro F1": r["Macro F1"],
        "Macro F1 Retention (%)": 100 * r["Macro F1"] / baseline["Macro F1"] if baseline["Macro F1"] else np.nan,
        "Inference Time (s)": r["Inference Time (s)"],
        "Prediction Throughput (sps)": r["Prediction Throughput (sps)"],
    })
df_channel_eff = pd.DataFrame(rows_ch)
df_channel_eff.to_csv(tables_dir / "channel_efficiency_v2.csv", index=False)
df_channel_eff.to_markdown(tables_dir / "channel_efficiency_v2.md", index=False)
display(df_channel_eff)


,Configuration,Features Used,Accuracy,Accuracy Retention (%),Macro F1,Macro F1 Retention (%),Inference Time (s),Prediction Throughput (sps)
0,"All Channels (12, 50 features)",50,0.455120,100.000000,0.171567,100.000000,1.013050,102373.001920
1,Top 8 Channels,47,0.453162,99.569915,0.170685,99.485791,1.066281,97262.343521
2,Top 6 Channels,39,0.443597,97.468220,0.147535,85.992349,1.505181,68901.363733
3,Top 4 Channels,25,0.428912,94.241525,0.124799,72.740418,1.009026,102781.299180
4,Top 2 Channels,15,0.400553,88.010593,0.067716,39.469082,1.418155,73129.546819
5,Single Best Channel,10,0.393717,86.508475,0.033945,19.785483,0.974779,106392.312644


## Section 4: Feature Family (Domain) Interpretation
Real accuracy when training on a single feature-family domain (Time / Frequency / Wavelet) or
pairwise combinations, versus the full 50-feature baseline.

In [5]:
family_order = [
    ("baseline_all_50_features", "All Feature Families (Baseline)"),
    ("time_domain_only", "Time Domain Only"),
    ("frequency_domain_only", "Frequency Domain Only"),
    ("wavelet_domain_only", "Wavelet Domain Only"),
    ("time_plus_frequency_domains", "Time + Frequency Domains"),
    ("time_plus_wavelet_domains", "Time + Wavelet Domains"),
    ("frequency_plus_wavelet_domains", "Frequency + Wavelet Domains"),
]
rows_fam = []
for cfg, label in family_order:
    r = df_all[df_all["Config"] == cfg].iloc[0]
    f1_drop = baseline["Macro F1"] - r["Macro F1"]
    rows_fam.append({
        "Feature Family Domain": label,
        "Feature Count": int(r["Feature Count"]),
        "Accuracy": r["Accuracy"],
        "Macro F1": r["Macro F1"],
        "F1 Drop vs Baseline": f1_drop,
        "Macro F1 Retention (%)": 100 * r["Macro F1"] / baseline["Macro F1"] if baseline["Macro F1"] else np.nan,
    })
df_family = pd.DataFrame(rows_fam)
df_family.to_csv(tables_dir / "feature_family_summary_v2.csv", index=False)
df_family.to_markdown(tables_dir / "feature_family_summary_v2.md", index=False)
display(df_family)


,Feature Family Domain,Feature Count,Accuracy,Macro F1,F1 Drop vs Baseline,Macro F1 Retention (%)
0,All Feature Families (Baseline),50,0.455120,0.171567,0.000000,100.000000
1,Time Domain Only,9,0.403379,0.066393,0.105174,38.698058
2,Frequency Domain Only,10,0.403610,0.090504,0.081064,52.751080
3,Wavelet Domain Only,31,0.441717,0.148058,0.023510,86.297216
4,Time + Frequency Domains,19,0.425884,0.128414,0.043154,74.847362
5,Time + Wavelet Domains,40,0.448254,0.160325,0.011242,93.447437
6,Frequency + Wavelet Domains,41,0.450231,0.166713,0.004854,97.170514


## Section 5: SHAP-vs-Ablation Consistency Check
Rather than a fabricated per-feature Spearman correlation (individual per-feature ablation was
not run -- only grouped top/bottom-N removal), this section reports the real, direct evidence
of consistency between the SHAP ranking (Notebook 12) and empirical impact: removing
SHAP-important (top-N) features causes a real, substantial, monotonically-growing F1 drop,
while removing SHAP-unimportant (bottom-N) features causes negligible or even slightly
positive change (consistent with those features being redundant/noisy).

In [6]:
top_removal = df_feat_removal[df_feat_removal["Configuration"].str.contains("Top")]
bottom_removal = df_feat_removal[df_feat_removal["Configuration"].str.contains("Bottom")]

consistency_summary = pd.DataFrame({
    "Removal Type": ["Top-N (SHAP-important)", "Bottom-N (SHAP-unimportant)"],
    "Mean F1 Drop": [top_removal["F1 Drop vs Baseline"].mean(), bottom_removal["F1 Drop vs Baseline"].mean()],
    "Mean Drop Per Feature": [top_removal["Drop Per Removed Feature"].mean(), bottom_removal["Drop Per Removed Feature"].mean()],
    "Monotonic with N Removed": [
        bool(np.all(np.diff(top_removal.sort_values("Features Removed")["Macro F1"].values) <= 1e-6)),
        None,
    ],
})
consistency_summary.to_csv(tables_dir / "xai_consistency_summary_v2.csv", index=False)
display(consistency_summary)
top_mean_drop = top_removal["F1 Drop vs Baseline"].mean()
bottom_mean_drop = bottom_removal["F1 Drop vs Baseline"].mean()
print(f"\nTop-N removal mean F1 drop: {top_mean_drop:.4f}")
print(f"Bottom-N removal mean F1 drop: {bottom_mean_drop:.4f}")
print(f"Ratio (top impact / bottom impact): {top_mean_drop / max(bottom_mean_drop, 1e-9):.1f}x")


,Removal Type,Mean F1 Drop,Mean Drop Per Feature,Monotonic with N Removed
0,Top-N (SHAP-important),0.030417,0.002331,True
1,Bottom-N (SHAP-unimportant),-0.002089,-0.000254,None



Top-N removal mean F1 drop: 0.0304
Bottom-N removal mean F1 drop: -0.0021
Ratio (top impact / bottom impact): 30416880.5x


## Section 6: Performance Degradation Curves
Macro F1 as a function of features retained, across the feature-removal and channel-reduction
sweeps.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(df_feat_removal["Features Retained"], df_feat_removal["Macro F1"], "o-", color="#4C72B0")
axes[0].axhline(baseline["Macro F1"], color="gray", linestyle="--", label="Baseline")
axes[0].set_xlabel("Features Retained"); axes[0].set_ylabel("Macro F1")
axes[0].set_title("Macro F1 vs Features Retained (SHAP removal sweep)")
axes[0].legend()

axes[1].plot(df_channel_eff["Features Used"], df_channel_eff["Macro F1"], "o-", color="#C44E52")
axes[1].axhline(baseline["Macro F1"], color="gray", linestyle="--", label="Baseline")
axes[1].set_xlabel("Features Used (channel-restricted)"); axes[1].set_ylabel("Macro F1")
axes[1].set_title("Macro F1 vs Channel Reduction")
axes[1].legend()

fig.tight_layout()
fig.savefig(figures_dir / "figure_13_01_performance_drop_curves_v2.png", dpi=150)
plt.close(fig)
print("Saved figure_13_01_performance_drop_curves_v2.png")


Saved figure_13_01_performance_drop_curves_v2.png


In [8]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(family_order) - 1)
labels = [l for _, l in family_order[1:]]
f1s = [df_all[df_all["Config"] == c].iloc[0]["Macro F1"] for c, _ in family_order[1:]]
ax.bar(x, f1s, color="#55A868")
ax.axhline(baseline["Macro F1"], color="gray", linestyle="--", label="Baseline (all 50 features)")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha="right")
ax.set_ylabel("Macro F1"); ax.set_title("Feature Family Domain Comparison")
ax.legend()
fig.tight_layout()
fig.savefig(figures_dir / "figure_13_02_feature_family_comparison_v2.png", dpi=150)
plt.close(fig)
print("Saved figure_13_02_feature_family_comparison_v2.png")


Saved figure_13_02_feature_family_comparison_v2.png


## Section 7: Deployment Trade-Off Recommendation
Real accuracy/latency/feature-count trade-offs across the channel-reduction sweep, for
selecting an embedded-hardware deployment profile.

In [9]:
deploy_rows = []
hw_targets = {
    "All Channels (12, 50 features)": "Workstation / Cloud API",
    "Top 8 Channels": "Moderate-Density Armband",
    "Top 6 Channels": "Standard Armband (Recommended)",
    "Top 4 Channels": "Low-Power Compact Armband",
    "Top 2 Channels": "Ultra-Minimal Patch",
    "Single Best Channel": "Single-Sensor Bio-Patch",
}
for cfg, label in channel_order:
    r = df_all[df_all["Config"] == cfg].iloc[0]
    deploy_rows.append({
        "Deployment Profile": label,
        "Features": int(r["Feature Count"]),
        "Accuracy": r["Accuracy"],
        "Macro F1": r["Macro F1"],
        "Macro F1 Retention (%)": 100 * r["Macro F1"] / baseline["Macro F1"] if baseline["Macro F1"] else np.nan,
        "Inference Time (s, full test set)": r["Inference Time (s)"],
        "Throughput (sps)": r["Prediction Throughput (sps)"],
        "Hardware Target": hw_targets[label],
    })
df_deploy = pd.DataFrame(deploy_rows)
df_deploy.to_csv(tables_dir / "deployment_recommendation_v2.csv", index=False)
df_deploy.to_markdown(tables_dir / "deployment_recommendation_v2.md", index=False)
display(df_deploy)


,Deployment Profile,Features,Accuracy,Macro F1,Macro F1 Retention (%),"Inference Time (s, full test set)",Throughput (sps),Hardware Target
0,"All Channels (12, 50 features)",50,0.455120,0.171567,100.000000,1.013050,102373.001920,Workstation / Cloud API
1,Top 8 Channels,47,0.453162,0.170685,99.485791,1.066281,97262.343521,Moderate-Density Armband
2,Top 6 Channels,39,0.443597,0.147535,85.992349,1.505181,68901.363733,Standard Armband (Recommended)
3,Top 4 Channels,25,0.428912,0.124799,72.740418,1.009026,102781.299180,Low-Power Compact Armband
4,Top 2 Channels,15,0.400553,0.067716,39.469082,1.418155,73129.546819,Ultra-Minimal Patch
5,Single Best Channel,10,0.393717,0.033945,19.785483,0.974779,106392.312644,Single-Sensor Bio-Patch


## Section 8: Publication Tables Exported
- `feature_removal_summary_v2.csv / md`
- `channel_efficiency_v2.csv / md`
- `feature_family_summary_v2.csv / md`
- `xai_consistency_summary_v2.csv`
- `deployment_recommendation_v2.csv / md`

(The original, non-`_v2` files under `outputs/tables/` were generated from tiny
non-representative training subsamples and are superseded by these.)

## Section 9 & 10: Manuscript Results & Discussion
Real journal report text generated from the numbers computed above, written to
`outputs/reports/`:
- **`results_notebook13_updated.md`**
- **`discussion_notebook13_updated.md`**

In [10]:
val_report = f"""# Ablation Study Validation Report (v2 - Real Full-Scale Training)
- **Classifier Model**: CATBOOST (Optimized, 308-trial retuned)
- **Total Configurations**: {len(df_all)} / 20 Verified
- **Training Split**: Real 484,700-row subject-disjoint split (28 subjects)
- **Test Split**: Real 103,709-row held-out split (6 subjects)
- **Training Platform**: Google Colab (T4 GPU, ~53s/config after CatBoost upgrade resolved
  an initial CUDA driver incompatibility; the baseline config ran before the fix, on CPU,
  in 5398s / ~90 minutes)
"""
(reports_dir / "validation_report_ablation_v2.md").write_text(val_report, encoding="utf-8")
print(val_report)


# Ablation Study Validation Report (v2 - Real Full-Scale Training)
- **Classifier Model**: CATBOOST (Optimized, 308-trial retuned)
- **Total Configurations**: 20 / 20 Verified
- **Training Split**: Real 484,700-row subject-disjoint split (28 subjects)
- **Test Split**: Real 103,709-row held-out split (6 subjects)
- **Training Platform**: Google Colab (T4 GPU, ~53s/config after CatBoost upgrade resolved
  an initial CUDA driver incompatibility; the baseline config ran before the fix, on CPU,
  in 5398s / ~90 minutes)



## Notebook Summary

### Executive Summary
This notebook ran and evaluated 20 real ablation configurations against the fully re-tuned
(308-trial Optuna) CatBoost classifier, each a genuine model retrained from scratch on the
real 484,700-row subject-disjoint training split restricted to that configuration's feature
subset, evaluated on the real 103,709-row held-out test split. All training was executed on
Google Colab (T4 GPU).

### Key Findings & Metrics (real, computed above)
- **Baseline (all 50 features)**: Accuracy = 45.51%, Macro F1 = 17.16% -- consistent with
  Notebook 10's held-out CatBoost result (45.42% / 16.55%) and Notebook 11's LOSO mean
  (45.67% / 16.43%), confirming this ablation baseline is measuring the same real model
  behavior as the rest of the project.
- **Feature removal is directionally consistent with the SHAP ranking**: removing the top-20
  SHAP-important features drops Macro F1 to 11.84% (a 31% relative drop), while removing the
  bottom-20 SHAP-unimportant features actually slightly *improves* Macro F1 to 17.29% (removing
  noisy/redundant features). This is real, substantial evidence that the Notebook 12 SHAP
  ranking reflects genuine predictive importance, not an artifact.
- **Channel reduction**: the top-8-channel subset (47 of 50 features) retains 99.5% of
  baseline Macro F1 (17.07% vs 17.16%); the top-6-channel subset (39 features) retains 86.0%
  (14.75%). Below that, performance degrades faster (top-4: 72.7% retention; top-2: 39.5%;
  single channel: 19.8%).
- **Feature family**: Wavelet-domain-only features (31 of 50) retain 86.3% of baseline Macro F1
  (14.81%) alone, confirming Notebook 12's finding that the wavelet domain dominates SHAP
  attribution (65.1% of total importance). Frequency-only and time-only are substantially
  weaker alone (9.05% and 6.64% Macro F1 respectively).

### Corrected Finding vs. Prior (Small-Sample) Results
The prior version of Notebook 13 read tables computed from 5,000-10,000-row training
subsamples, producing three different, mutually inconsistent "baseline" accuracy figures
(41.43%, 38.21%, 24.18%) across its feature-removal, channel-efficiency, and deployment
tables respectively -- none matching the real ~45% baseline established throughout this
project (Notebooks 10 and 11). This rebuild uses the full real training split for every
configuration, and the baseline figure (45.51%) is now consistent across every table and
with the rest of the project.

### Publication Artifacts Created
- **Tables** (`outputs/tables/`): `feature_removal_summary_v2.csv/md`, `channel_efficiency_v2.csv/md`,
  `feature_family_summary_v2.csv/md`, `xai_consistency_summary_v2.csv`, `deployment_recommendation_v2.csv/md`
- **Figures** (`outputs/figures/`): `figure_13_01_performance_drop_curves_v2.png`,
  `figure_13_02_feature_family_comparison_v2.png`
- **Reports** (`outputs/reports/`): `results_notebook13_updated.md`, `discussion_notebook13_updated.md`,
  `validation_report_ablation_v2.md`

### Recommendations for Notebook 14 (Deployment & Model Optimization)
The top-8-channel, 47-feature configuration offers the best accuracy/hardware trade-off
(99.5% Macro F1 retention with 2 fewer channels); the top-6-channel, 39-feature configuration
is a reasonable lower-power alternative at a real, disclosed 86.0% F1 retention cost -- not
the previously-claimed near-lossless "98.6% retention" figure from the small-sample tables.